# Recupero Dataset

In [213]:
!curl -o contatti.json https://proai-datasets.s3.eu-west-3.amazonaws.com/contatti.json

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  3677  100  3677    0     0  13470      0 --:--:-- --:--:-- --:--:-- 13568


# Installo pacchetti necessari

In [214]:
%pip install pymongo

Note: you may need to restart the kernel to use updated packages.


# Gestisco import

In [215]:
from pymongo import MongoClient
import json
from pymongo import ReturnDocument

# Connessione al cluster

In [216]:
client = MongoClient("mongodb://mongo1:27017,mongo2:27018,mongo3:27019/?replicaSet=rs0")

# Setup Database

In [217]:
client.drop_database('contatti')
contatti_db = client['contatti']
contatti = contatti_db['contatti']
contatti.drop()
with open('contatti.json') as f:
    d = json.load(f)
    contatti.insert_many(d)

# Svolgimento del progetto

# Considerazioni Personali

Come da specifiche creo un indice univolo per garantire l'integrità dei dati

In [205]:
contatti.create_index(
    { "Nome": 1, "Cognome": 1 },
    unique = True
)

'Nome_1_Cognome_1'

## Interrogazione dei Dati

### Trovare tutti i contatti associati alla società WebCorp.

Ho utilizzato semplicemente un find:

In [206]:
list(contatti.find({'Società': 'WebCorp'}))

[{'_id': ObjectId('68333a53fe11e982b479bac8'),
  'Nome': 'Marco',
  'Cognome': 'Bianchi',
  'Numero_di_cellulare': '348123456',
  'Società': 'WebCorp',
  'Data_di_compleanno': '1985-03-22',
  'Tag': ['lavoro', 'tech'],
  'Altri_contatti': {'Email': ['marco.bianchi@example.com',
    'marco.b@webcorp.it'],
   'Indirizzo': 'Via Roma 10, Milano'},
  'Chiamate_ultimo_mese': 8,
  'Amici_stretti': False},
 {'_id': ObjectId('68333a53fe11e982b479bad2'),
  'Nome': 'Alessia',
  'Cognome': 'Moretti',
  'Numero_di_cellulare': ['347098765', '321654987'],
  'Società': 'WebCorp',
  'Tag': ['lavoro', 'viaggi'],
  'Altri_contatti': {'Email': ['alessia.moretti@example.com',
    'sales@moretti.com']},
  'Chiamate_ultimo_mese': 25,
  'Amici_stretti': True}]

### Identificare i contatti con più di un numero di telefono.

Uso $cond come un case..when in sql per distinguere Array dalle stringe per non ottenere errore:

In [207]:
list(contatti.find(
    {
        "$expr": {
            "$gt": [
                { "$cond": [
                    { "$isArray": "$Numero_di_cellulare" },
                    { "$size": "$Numero_di_cellulare" },
                    0
                ]},
                1
            ]
        }
    },
    {
        "_id": 0,
        "Nome": 1, 
        "Cognome": 1,           
    }
))

[{'Nome': 'Laura', 'Cognome': 'Rossi'},
 {'Nome': 'Chiara', 'Cognome': 'Marroni'},
 {'Nome': 'Alessia', 'Cognome': 'Moretti'}]

### Estrarre solo i numeri di telefono dei contatti con tag “lavoro”.

Utilizzo Aggregate invece di find perchè consente anche di strasformare i dati attraverso degli stage invece di
filtrarli e proiettarli solamente. Nel caso specifico l'ho usato per appiattire i numeri di telefono nel caso
ve ne sia più d'uno in una lista interna al documento.

In [208]:
list(contatti.aggregate([
    {"$match":{"Tag":{"$in":["lavoro"]}}},
    {"$project":{"_id":0,"Numero_di_cellulare":1}},
    {"$unwind":"$Numero_di_cellulare"}
    ]))

[{'Numero_di_cellulare': '348123456'},
 {'Numero_di_cellulare': '349654321'},
 {'Numero_di_cellulare': '347098765'},
 {'Numero_di_cellulare': '321654987'}]

### Riportare nome e cognome dei contatti senza propri social.

Filtro per esistenza del campo profilo social e poi proietto nome e cognome.

In [209]:
list(contatti.find({"Altri_contatti.Profilo_social":{"$exists":True}},{"_id":0,"Nome":1,"Cognome":1}))

[{'Nome': 'Laura', 'Cognome': 'Rossi'},
 {'Nome': 'Sara', 'Cognome': 'Gialli'},
 {'Nome': 'Elena', 'Cognome': 'Viola'}]

### Contare quanti contatti sono etichettati come “amici stretti” e quanti no.

Per farlo utilizzo aggregate. Inizialmente proietto solo il campo amici stretti.
Successivamente gruppo per tale campo ed effettuo il count per valore.
Infine effettuo il sort. Non filtro o verifico la presenza del campo per intercettare
eventuali mancanze di valore da riportare come None. Inizialmente avevo pensato di considerarli
Falsi Ma dopo averci riflettuto l'assenza di informazione è più appropriata.

In [210]:
list(contatti.aggregate([
    { "$project": { "Amici_stretti": 1 } },
    {"$group":{
            "_id":"$Amici_stretti",
            "conteggio": { "$count": { } }
            }
        },
    { "$sort": { "_id": -1 } }
    ]))

[{'_id': True, 'conteggio': 5}, {'_id': False, 'conteggio': 6}]

### Calcolare il numero medio di chiamate effettuate nell’ultimo mese dai contatti “amici stretti”.

## Aggiornamenti ai Dati

### Aggiungi a Simone Azzurri il numero 345678902

Per il caso specifico utilizzo un file and update,
utilizzando il campo del documento presente e unendolo al nuovo numero in una lista
per mantenere una struttura omologa ad altri documenti con più numeri di telefono.

In [211]:
contatti.find_one_and_update(
    filter={
        "Nome": "Simone",
        "Cognome": "Azzurri"
    },
    update=[ {
        "$set": {
            "Numero_di_cellulare": ["$Numero_di_cellulare","345678902"],
        }
    }] ,
    upsert=True,
    return_document=ReturnDocument.AFTER
)

{'_id': ObjectId('68333a53fe11e982b479bad1'),
 'Nome': 'Simone',
 'Cognome': 'Azzurri',
 'Numero_di_cellulare': ['345678901', '345678902'],
 'Data_di_compleanno': '1979-05-30',
 'Tag': ['famiglia', 'viaggi'],
 'Altri_contatti': {'Email': 'simone.azzurri@example.net'},
 'Chiamate_ultimo_mese': 20,
 'Amici_stretti': True}

### Aggiungi un nuovo documento contenente il contatto di Mary Salgado con numero 346679933 e indirizzo Via 25 Aprile 3, Firenze

Per svolgere questo esercizio utilizzo un upsert invece di una semplice insert 
per assicurarmi di non avere duplicazioni accidentali. Per il filtro mi baso sui 
campi che compongono l'indice univoco da me creato.

In [212]:
contatti.find_one_and_update(
    filter={
        "Nome": "Mary",
        "Cognome": "Salgado"
    },
    update={
        "$set": {
            "Numero_di_cellulare": "346679933",
            "Altri_contatti.Indirizzo": "Via 25 Aprile 3, Firenze"
        }
    },
    upsert=True,
    return_document=ReturnDocument.AFTER
)

{'_id': ObjectId('68333a53328e6f1691df4a3a'),
 'Cognome': 'Salgado',
 'Nome': 'Mary',
 'Altri_contatti': {'Indirizzo': 'Via 25 Aprile 3, Firenze'},
 'Numero_di_cellulare': '346679933'}